In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv('netflix_titles.csv')
print(df.shape)
df.head
df

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include="object")

In [ ]:
df.columns.tolist()

In [ ]:
df.sample(5)

In [ ]:
print({df.shape[0]})
print({df.shape[1]})
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().mean() * 100

In [ ]:
df.duplicated()

In [ ]:
df.memory_usage(deep=True)

In [ ]:
print('Доля типов контента:')
print(df['type'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

** **
РАСШИРЕННАЯ СТАТИСТИКА
** **

In [ ]:
df["release_year"].agg([
    "min",
    "max",
    "mean",
    "median",
    "var",
    "skew",
    "kurt",
    "std"
    ])

In [ ]:
df['release_year'].mode()

In [ ]:
df['release_year'].quantile([0.05, 0.25, 0.5, 0.75, 0.95])

** **
ЭНКОДИНГ
** **

** before **

In [ ]:
print(df["type"].head())

In [ ]:
df_encoded = pd.get_dummies(df, columns=['type'])

** after **

In [ ]:
print(df_encoded[["type_Movie", "type_TV Show"]].head())

In [ ]:
before = df[["type"]].head()
after = pd.get_dummies(df["type"]).head()

print("Before Encoding:")
display(before)

print("After Encoding:")
display(after)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output = False)

encoded = encoder.fit_transform(df[['type']])
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(['type'])
)
encoded_df.head()


** FEATURE HASHING **

In [ ]:
from sklearn.feature_extraction import FeatureHasher

hasher = FeatureHasher(n_features=10, input_type="string")
director_list = df["director"].fillna("Unknown").apply(lambda x: [x])
hashed_features = hasher.transform(director_list)

hashed_df = pd.DataFrame(
    hashed_features.toarray(),
    columns=[f"director_hash_{i}" for i in range(10)]
)

hashed_df.head()


** **
VISUALIZATION
** **


In [ ]:
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(df["release_year"], bins=30, color= "red", edgecolor="white")

plt.title('Распределение по году выхода')
plt.xlabel('Год')
plt.ylabel('Количество')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

# hue='type' — разные цвета для Movie и TV Show
sns.kdeplot(data=df, x='release_year', hue='type', fill=True, alpha=0.5)

plt.title('Content release year')
plt.show()

In [ ]:
#seaborn
movies = df[df['type'] == 'Movie'].copy()
movies['duration_value'] = movies['duration'].str.extract(r'(\d+)').astype(float)

plt.figure(figsize=(10, 5))

sns.scatterplot(data=movies, x='release_year', y='duration_value', alpha=0.4)

plt.title('release year & movie duration')
plt.xlabel('year')
plt.ylabel('minutes')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.boxplot(movies['duration_value'].dropna())

plt.title('movie duration')
plt.ylabel('minutes')
plt.show()

In [ ]:
#barplot
#там был мусор в виде минут каких то, их убираем сначала и потом строим график

print(df['rating'].unique())
valid_ratings = ['G', 'PG', 'PG-13', 'R', 'NC-17',
                 'TV-Y', 'TV-Y7', 'TV-Y7-FV', 'TV-G', 
                 'TV-PG', 'TV-14', 'TV-MA', 'NR', 'UR']

df = df[df['rating'].isin(valid_ratings)]
print(df['rating'].unique())

#####################


plt.figure(figsize=(10, 5))

order = df['rating'].value_counts().index
sns.countplot(data=df, y='rating', order=order, palette='Reds_r')

plt.title('rating quantity')
plt.show()

In [ ]:
print(df.select_dtypes(include='number').columns)

df['duration_value'] = df['duration'].str.extract(r'(\d+)').astype(float)

df['year_added'] = pd.to_datetime(df['date_added'], errors='coerce').dt.year
df['month_added'] = pd.to_datetime(df['date_added'], errors='coerce').dt.month

print(df.select_dtypes(include='number').columns)


In [ ]:
numeric_df = df.select_dtypes(include='number')

plt.figure(figsize=(8, 6))

sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')

plt.title('correlation heatmap ')
plt.show()

самая заметная связь — release_year и duration_value: -0.25
это значит, что старые фильмы в среднем чуть длиннее новых, но связь слабая, делать выводы на её основе не стоит

все остальные числа близки к нулю — сильных зависимостей нет


** итоговые выводы **

- на Netflix фильмов гораздо больше чем сериалов, примерно 60% на 40%
- большинство контента появилось после 2015 года так как платформа резко начала расти
- США производят больше всего контента, за ними идут Индия и Великобритания
- самые популярные рейтинги — TV-MA и TV-14,то есть контент в основном для взрослых
- у колонки director очень много пропусков, что это нормально для сериалов, где режиссёров несколько
- Средняя продолжительность фильма около 100 минут, но есть выбросы — фильмы по 3+ часа
- Контент добавляется на платформу неравномерно — есть месяцы с явными пиками
- Большинство фильмов попадают на Netflix через 2-5 лет после выхода в прокат
- Детского контента (TV-G, G) на платформе заметно меньше чем взрослого
